# Bengaluru House Prices

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK


### 2. Load Data

In [2]:
df_house = pd.read_csv('bengaluru_house_prices.csv')
df_house.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


### 3. Data Cleaning

In [3]:
def convert_sqft_to_num(x):
    try:
        if '-' in str(x):
            low, high = str(x).split('-')
            return (float(low) + float(high)) / 2
        return float(x)
    except:
        return np.nan

df_house['total_sqft'] = df_house['total_sqft'].apply(convert_sqft_to_num)
df_house['bhk'] = df_house['size'].astype(str).str.extract(r'(\d+)').astype(float)

df_house = df_house.drop(columns=['society', 'availability', 'size'])
df_house = df_house.dropna(subset=['total_sqft', 'bhk', 'price', 'location'])

df_house = df_house[~(df_house['total_sqft'] / df_house['bhk'] < 300)]

df_house['price_per_sqft'] = df_house['price'] * 100000 / df_house['total_sqft']

df_house['location'] = df_house['location'].apply(lambda x: str(x).strip())
location_counts = df_house['location'].value_counts()
rare_locations = location_counts[location_counts <= 10].index
df_house['location'] = df_house['location'].apply(lambda x: 'other' if x in rare_locations else x)

print("Shape after basic cleaning:", df_house.shape)
df_house.head()

Shape after basic cleaning: (12513, 8)


,area_type,location,total_sqft,bath,balcony,price,bhk,price_per_sqft
0,Super built-up Area,Electronic City Phase II,1056.0,2.0,1.0,39.07,2.0,3699.810606
1,Plot Area,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4.0,4615.384615
2,Built-up Area,Uttarahalli,1440.0,2.0,3.0,62.00,3.0,4305.555556
3,Super built-up Area,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3.0,6245.890861
4,Super built-up Area,Kothanur,1200.0,2.0,1.0,51.00,2.0,4250.000000


#### 3.1 Remove Price-per-Square-Foot Outliers (for Each Location)

In [4]:
def remove_pps_outliers(df):
    filtered = pd.DataFrame()
    for location, subdf in df.groupby('location'):
        mean_pps = np.mean(subdf.price_per_sqft)
        std_pps = np.std(subdf.price_per_sqft)
        reduced = subdf[(subdf.price_per_sqft > (mean_pps - std_pps)) &
                         (subdf.price_per_sqft <= (mean_pps + std_pps))]
        filtered = pd.concat([filtered, reduced], ignore_index=True)
    return filtered

df_house = remove_pps_outliers(df_house)
print("Shape after price-per-sqft outlier removal:", df_house.shape)

Shape after price-per-sqft outlier removal: (10315, 8)


#### 3.2 Remove BHK Outliers (for Each Location)

In [5]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk - 1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df.price_per_sqft < stats['mean']].index.values
                )
    return df.drop(exclude_indices, axis='index')

df_house = remove_bhk_outliers(df_house)
df_house = df_house.drop(columns=['price_per_sqft'])
print("Final shape after cleaning:", df_house.shape)

Final shape after cleaning: (7289, 7)


### 4. Prepare Features & Target

In [ ]:
df_house_encoded = pd.get_dummies(df_house, columns=['area_type', 'location'], drop_first=True)

X_house = df_house_encoded.drop(columns=['price'])
y_house = df_house_encoded['price']

X_house = X_house.fillna(X_house.median())

print("X_house shape:", X_house.shape)

X_house shape: (7289, 229)


### 5. Train / Test Split

In [7]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

X_train_h_opt, X_valid_h_opt, y_train_h_opt, y_valid_h_opt = train_test_split(
    X_train_h, y_train_h, test_size=0.2, random_state=42
)

### 6. Find Best Parameters (Optuna)

In [8]:
def house_objective_optuna(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train_h_opt, y_train_h_opt)

    preds = model.predict(X_valid_h_opt)
    return r2_score(y_valid_h_opt, preds)

study_house = optuna.create_study(direction='maximize')
study_house.optimize(house_objective_optuna, n_trials=30)

print("Best Optuna R2 (validation):", round(study_house.best_value, 4))
print("Best Optuna Params:", study_house.best_params)

[I 2026-08-14 20:55:13,873] A new study created in memory with name: no-name-2d119f69-17ac-4791-bca2-ada7d765f4e9
[I 2026-08-14 20:55:17,548] Trial 0 finished with value: 0.7845924627266696 and parameters: {'n_estimators': 351, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7845924627266696.
[I 2026-08-14 20:55:21,587] Trial 1 finished with value: 0.7842709714182422 and parameters: {'n_estimators': 461, 'max_depth': 24, 'min_samples_split': 9, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7845924627266696.
[I 2026-08-14 20:55:23,330] Trial 2 finished with value: 0.7921452666494543 and parameters: {'n_estimators': 308, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 4}. Best is trial 2 with value: 0.7921452666494543.
[I 2026-08-14 20:55:25,735] Trial 3 finished with value: 0.7792811006588236 and parameters: {'n_estimators': 433, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 4}. Best is trial 2 with value:

Best Optuna R2 (validation): 0.8808
Best Optuna Params: {'n_estimators': 354, 'max_depth': 27, 'min_samples_split': 5, 'min_samples_leaf': 1}


### 7. Find Best Parameters (Hyperopt)

In [9]:
def house_objective_hyperopt(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_samples_split'] = int(params['min_samples_split'])
    params['min_samples_leaf'] = int(params['min_samples_leaf'])

    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_h_opt, y_train_h_opt)

    preds = model.predict(X_valid_h_opt)
    score = r2_score(y_valid_h_opt, preds)

    return {'loss': -score, 'status': STATUS_OK}

house_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 1),
    'max_depth': hp.quniform('max_depth', 5, 30, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1)
}

house_trials = Trials()

best_house_hyperopt = fmin(
    fn=house_objective_hyperopt,
    space=house_space,
    algo=tpe.suggest,
    max_evals=30,
    trials=house_trials,
    rstate=np.random.default_rng(42)
)

best_house_hyperopt_params = {
    'n_estimators': int(best_house_hyperopt['n_estimators']),
    'max_depth': int(best_house_hyperopt['max_depth']),
    'min_samples_split': int(best_house_hyperopt['min_samples_split']),
    'min_samples_leaf': int(best_house_hyperopt['min_samples_leaf'])
}

print("Best Hyperopt Params:", best_house_hyperopt_params)

100%|██████████| 30/30 [01:31<00:00,  3.05s/trial, best loss: -0.8820998317270214]
Best Hyperopt Params: {'n_estimators': 499, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 1}


### 8. Best Parameters & Model Evaluation

In [10]:
optuna_house_model = RandomForestRegressor(**study_house.best_params, random_state=42, n_jobs=-1)
optuna_house_model.fit(X_train_h, y_train_h)
optuna_pred_h = optuna_house_model.predict(X_test_h)

print("Optuna House Model")
print("Test R2 Score:", round(r2_score(y_test_h, optuna_pred_h), 4))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test_h, optuna_pred_h)), 2))

print()

hyperopt_house_model = RandomForestRegressor(**best_house_hyperopt_params, random_state=42, n_jobs=-1)
hyperopt_house_model.fit(X_train_h, y_train_h)
hyperopt_pred_h = hyperopt_house_model.predict(X_test_h)

print("Hyperopt House Model")
print("Test R2 Score:", round(r2_score(y_test_h, hyperopt_pred_h), 4))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test_h, hyperopt_pred_h)), 2))

Optuna House Model
Test R2 Score: 0.7557
Test RMSE: 48.03

Hyperopt House Model
Test R2 Score: 0.7657
Test RMSE: 47.04


# Churn Modelling

### 1. Import Libraries

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

### 2. Load Data

In [12]:
df_churn = pd.read_csv('Churn_Modelling.csv')
df_churn.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### 3. Data Cleaning & Preparation

In [13]:
df_churn = df_churn.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df_churn_encoded = pd.get_dummies(df_churn, columns=['Geography', 'Gender'], drop_first=True)

X_churn = df_churn_encoded.drop(columns=['Exited'])
y_churn = df_churn_encoded['Exited']

### 4. Train / Test Split

In [14]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)

X_train_c_opt, X_valid_c_opt, y_train_c_opt, y_valid_c_opt = train_test_split(
    X_train_c, y_train_c, test_size=0.2, random_state=42, stratify=y_train_c
)

### 5. Find Best Parameters (Optuna)

In [15]:
def churn_objective_optuna(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train_c_opt, y_train_c_opt)

    preds = model.predict(X_valid_c_opt)
    return accuracy_score(y_valid_c_opt, preds)

study_churn = optuna.create_study(direction='maximize')
study_churn.optimize(churn_objective_optuna, n_trials=30)

print("Best Optuna Accuracy (validation):", round(study_churn.best_value, 4))
print("Best Optuna Params:", study_churn.best_params)

[I 2026-08-14 20:58:59,419] A new study created in memory with name: no-name-3e3bff96-7610-4287-85ea-3375591f62ea
[I 2026-08-14 20:59:00,619] Trial 0 finished with value: 0.85625 and parameters: {'n_estimators': 342, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.85625.
[I 2026-08-14 20:59:01,728] Trial 1 finished with value: 0.860625 and parameters: {'n_estimators': 314, 'max_depth': 14, 'min_samples_split': 10, 'min_samples_leaf': 5}. Best is trial 1 with value: 0.860625.
[I 2026-08-14 20:59:03,155] Trial 2 finished with value: 0.859375 and parameters: {'n_estimators': 428, 'max_depth': 25, 'min_samples_split': 10, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.860625.
[I 2026-08-14 20:59:03,840] Trial 3 finished with value: 0.860625 and parameters: {'n_estimators': 194, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.860625.
[I 2026-08-14 20:59:04,731] Trial 4 finished with value: 0.

Best Optuna Accuracy (validation): 0.865
Best Optuna Params: {'n_estimators': 117, 'max_depth': 23, 'min_samples_split': 5, 'min_samples_leaf': 4}


### 6. Find Best Parameters (Hyperopt)

In [16]:
def churn_objective_hyperopt(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_samples_split'] = int(params['min_samples_split'])
    params['min_samples_leaf'] = int(params['min_samples_leaf'])

    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_c_opt, y_train_c_opt)

    preds = model.predict(X_valid_c_opt)
    score = accuracy_score(y_valid_c_opt, preds)

    return {'loss': -score, 'status': STATUS_OK}

churn_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 1),
    'max_depth': hp.quniform('max_depth', 3, 25, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1)
}

churn_trials = Trials()

best_churn_hyperopt = fmin(
    fn=churn_objective_hyperopt,
    space=churn_space,
    algo=tpe.suggest,
    max_evals=30,
    trials=churn_trials,
    rstate=np.random.default_rng(42)
)

best_churn_hyperopt_params = {
    'n_estimators': int(best_churn_hyperopt['n_estimators']),
    'max_depth': int(best_churn_hyperopt['max_depth']),
    'min_samples_split': int(best_churn_hyperopt['min_samples_split']),
    'min_samples_leaf': int(best_churn_hyperopt['min_samples_leaf'])
}

print("Best Hyperopt Params:", best_churn_hyperopt_params)

100%|██████████| 30/30 [00:36<00:00,  1.21s/trial, best loss: -0.864375]
Best Hyperopt Params: {'n_estimators': 335, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 4}


### 7. Best Parameters & Model Evaluation

In [17]:
optuna_churn_model = RandomForestClassifier(**study_churn.best_params, random_state=42, n_jobs=-1)
optuna_churn_model.fit(X_train_c, y_train_c)
optuna_pred_c = optuna_churn_model.predict(X_test_c)

print("Optuna Churn Model")
print("Test Accuracy:", round(accuracy_score(y_test_c, optuna_pred_c), 4))
print(classification_report(y_test_c, optuna_pred_c))

print()

hyperopt_churn_model = RandomForestClassifier(**best_churn_hyperopt_params, random_state=42, n_jobs=-1)
hyperopt_churn_model.fit(X_train_c, y_train_c)
hyperopt_pred_c = hyperopt_churn_model.predict(X_test_c)

print("Hyperopt Churn Model")
print("Test Accuracy:", round(accuracy_score(y_test_c, hyperopt_pred_c), 4))
print(classification_report(y_test_c, hyperopt_pred_c))

Optuna Churn Model
Test Accuracy: 0.8665
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.79      0.46      0.59       407

    accuracy                           0.87      2000
   macro avg       0.84      0.72      0.75      2000
weighted avg       0.86      0.87      0.85      2000


Hyperopt Churn Model
Test Accuracy: 0.868
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.81      0.46      0.58       407

    accuracy                           0.87      2000
   macro avg       0.84      0.72      0.75      2000
weighted avg       0.86      0.87      0.85      2000

